### Build testgen Python Package

In [1]:
!pip -q install -e ../.

### Imports

In [1]:
import pandas as pd
import json
from tqdm.notebook import tqdm
from testgen.utils import *
from pathlib import Path
from datetime import datetime as dt
import os

### EDA

In [2]:
base_path = Path().cwd().parent

In [3]:
# # read data and rename columns
# df = pd.read_csv(base_path / "data/requirements_full.csv")
# df.columns = ["requirement", "s1", "s2", "s3", "s4", "s5", "s6"]
df = pd.read_excel(base_path / "data/requirements.xlsx")

In [4]:
# make sure all columns except the first one are binary
for c in df.columns[1:]:
    assert df[c].drop_duplicates().shape[0] == 2

# Find Examples

In [5]:
N_EXAMPLES = 1

indexes_to_drop, examples = get_examples_from_df(df, N_EXAMPLES)

In [6]:
# join all examples in a text format to add to prompt
examples_txt = ""

for e1 in examples.values():
    for e2 in e1:
        examples_txt += f"Requirement: {e2[0]}\n"
        examples_txt += f"Vector: {e2[1]}\n"
        examples_txt += f"Target Sensor/s: {' and '.join(df.columns[1:][[True if x == 1 else False for x in map(int, e2[1][1:-1].split(','))]])}\n"
        examples_txt += "\n"

In [7]:
print("\n".join(examples_txt.split("\n")[-9:]))

Requirement: The system must be able to identify a fault in the steering torque sensor and transition to a safe state without compromising vehicle control
Vector: [0,0,0,0,1]
Target Sensor/s: steering_torque

Requirement: The system must compensate for crosswinds maintain the intended cornering path based on the value of wheel steering angle and acceleration input
Vector: [1,1,0,0,0]
Target Sensor/s: acceleration_pedal and wheel_steering_angle




# LLM

In [9]:
from testgen.prompts import SystemPrompt
from testgen.prompts import Sensors
from testgen.prompts import UserPromptBulk

In [10]:
client = openai_client(
    "azure",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    # api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    api_version='2024-08-01-preview',
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
)

In [11]:
# t = client.chat.completions.create(
#     model="gpt-4o-mini", messages=[{"role": "user", "content": "Respond: Here"}]
# )
# t.choices[0].message.content

In [12]:
# print(SystemPrompt.format(sensors=Sensors,examples=examples_txt))

### Get Requirements for multi prediction

In [ ]:
def get_batches(SAMPLE_TYPE="random", N_REQS=5):
    batches = []

    while df.shape[0] > N_REQS:
        if SAMPLE_TYPE == "random":
            instances = df.sample(N_REQS)
            df.drop(index=instances.index, inplace=True)
        else:
            instances = df.iloc[:N_REQS, :]
            df.drop(index=instances.index, inplace=True)

        batches.append(instances)

    print("Number of Batches:", len(batches))
    print("Number of Instances left:", df.shape[0])

    return batches


def requirement_text(batches):
    # split and format batches
    for i, batch in enumerate(batches):
        idx = batch.index.to_list()
        req = batch["requirement"].tolist()
        vec = (
            batch.iloc[:, 1:]
            .apply(lambda x: "[" + ",".join(map(str, x)) + "]", axis=1)
            .to_list()
        )
        batches[i] = (idx, req, vec)

    # requirement text
    req_texts = []
    for i, batch in enumerate(batches):
        req_text = ""
        for i, r in enumerate(batch[1]):
            req_text += f"Requirement {i+1}: {r}.\n"
        req_texts.append(req_text)

    return batches, req_texts

In [18]:
df = pd.read_excel(base_path / "data/requirements.xlsx")
df.drop(index=indexes_to_drop, inplace=True)

N_REQS = 5
SAMPLE_TYPE = "random"

batches = get_batches(SAMPLE_TYPE, N_REQS)
batches, req_texts = requirement_text(batches)

Number of Batches: 37
Number of Instances left: 4


In [19]:
def invoke_bulk(model_name, temperature, client, batch,
                SystemPrompt, Sensors, examples_txt, UserPromptBulk, req_text):
    result = {}

    messages = [
        {
            "role": "system",
            "content": SystemPrompt.format(sensors=Sensors, examples=examples_txt),
        }
    ]

    # add user prompt
    messages.append({"role": "user", "content": UserPromptBulk.format(req=req_text)})

    start_time = time.perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=temperature,
    )
    response_time = round(time.perf_counter() - start_time, 6)

    vectors = [parse_result(v) for v in response.choices[0].message.content.split("\n")]
    accuracy = [True if gt == v else False for gt, v in zip(batch[2], vectors)]

    result["idx"] = batch[0]
    result["requirement"] = batch[1]
    result["true_vector"] = batch[2]
    result["ai_response"] = response.choices[0].message.content
    result["pred_vector"] = vectors
    result["response_time"] = response_time

    result["accuracy"] = accuracy

    result.update(response.usage.to_dict())

    return result

In [20]:
model_name = "gpt-4o-mini"
temperature = 0.0

results = []

for batch, req_text in zip(batches, req_texts):
    result = invoke_bulk(model_name, temperature, client, batch,
                         SystemPrompt, Sensors, examples_txt, UserPromptBulk, req_text)
    results.append(result)

    break

In [22]:
total_number_of_instances = len(results) * N_REQS

accuracy = 0
total_tokens = 0
total_completion_tokens = 0
total_time = 0

for r in results:
    accuracy += sum(r["accuracy"])
    total_tokens += r["total_tokens"]
    total_completion_tokens += r["completion_tokens"]
    total_time += r['response_time']

number_of_reqs = len(results)
accuracy /= total_number_of_instances
avg_time_per_req = round(total_time / total_number_of_instances, 6)
avg_token_per_req = total_tokens / total_number_of_instances
avg_completion_token_per_req = total_completion_tokens / total_number_of_instances

In [23]:
print("number_of_reqs:", number_of_reqs)
print("accuracy:", accuracy)
print("avg_time_per_req:", avg_time_per_req)
print("avg_token_per_req:", avg_token_per_req)
print("avg_completion_token_per_req:", avg_completion_token_per_req)

number_of_reqs: 1
accuracy: 0.8
avg_time_per_req: 0.201869
avg_token_per_req: 197.0
avg_completion_token_per_req: 16.0


# Save conversation

In [24]:
# save results
time = dt.now()

results_path = "results/bulk-{bulk}_{model}_n-{examples}_acc-{accuracy}_{time}.json"
results_path = results_path.format(
    model=model_name,
    examples=N_EXAMPLES,
    bulk=N_REQS,
    time=time.strftime('%m.%d.%Y-%H:%M:%S'),
    accuracy=round(accuracy, 3)
)

results_file = base_path / results_path
results_file.parent.mkdir(exist_ok=True)
results_file.touch()

with results_file.open("w") as f:
    json.dump({"accuracy": accuracy,
        "number_of_reqs": number_of_reqs,
        "total_tokens": total_tokens,
        "total_completion_tokens": total_completion_tokens,
        "avg_token_per_req": avg_token_per_req,
        "avg_completion_token_per_req": avg_completion_token_per_req,
        "avg_time_per_req": avg_time_per_req,
        "examples": examples,
        "responses": results}, f, indent=4)